# turkish_distill: teacher encoding

Thin notebook. Encodes a corpus with one or more teachers into the shard cache on Drive (`cache/teachers/<teacher>/<corpus fingerprint>/`). All logic lives in `src/turkish_distill`.

**Two notebooks at once:** open this notebook in two Colab runtimes and give each a different `TEACHERS` entry (for example `['bge_m3']` in one and `['qwen3_8b']` in the other). Every teacher has its own cache folder and shards are written atomically, so they do not interfere. If both are pointed at the same teacher, the second one stops with a message instead of repeating the first one's work. Encoding is resumable: after a disconnect, run the cells again and it continues at the next missing shard.

The corpus and the teacher definitions (model, revision, vector size) come from the config; `CORPUS_PATH` below overrides the corpus. The corpus is not changed here, and a cache belongs to exactly one corpus: a different text list gets a different fingerprint and a separate folder.

## 1. Setup (Drive, repo, install)

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # needed for gated models only
except Exception as exc:
    print('HF_TOKEN not set:', exc)

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

## 2. Choose what this notebook encodes

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import glob, importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# --- What to encode ---
CONFIG = 'configs/stage1_two_qwen3_8b_bge_m3.yaml'   # defines the teachers and the default corpus
TEACHERS = ['bge_m3']    # teacher names (from the config) THIS notebook encodes; use a different one in each notebook
CORPUS_PATH = None       # None: the config's corpus. Or a .jsonl on Drive with a "text" field (see the list below)
MAX_SAMPLES = None       # None: the config's value (stage 1: all texts). A number takes a seeded random subset
IGNORE_LOCK = False      # True only if the notebook that held the lock has died (see the message it prints)

def ov(key, value):
    return f'{key}={"null" if value is None else value}'

OVERRIDES = []
if CORPUS_PATH is not None:
    OVERRIDES.append(ov('data.corpus_path', CORPUS_PATH))
if MAX_SAMPLES is not None:
    OVERRIDES.append(ov('data.max_samples', MAX_SAMPLES))
override_args = ' '.join(f'--override {o}' for o in OVERRIDES)
only_args = ' '.join(f'--only {t}' for t in TEACHERS)

from turkish_distill.config import load_config
cfg = load_config(CONFIG, OVERRIDES)
unknown = [t for t in TEACHERS if t not in {x.name for x in cfg.teachers}]
if unknown:
    raise ValueError(f'{unknown} are not teachers of {CONFIG}: {[x.name for x in cfg.teachers]}')
if not os.path.exists(cfg.data.corpus_path):
    raise FileNotFoundError(f'{cfg.data.corpus_path} not found')

print('config :', CONFIG)
print('corpus :', cfg.data.corpus_path, '| max_samples:', cfg.data.max_samples, '| min_chars:', cfg.data.min_chars)
print('teachers here:', TEACHERS, '| all in config:', [x.name for x in cfg.teachers])
print('\ncorpora on Drive:')
for path in sorted(glob.glob(f'{DRIVE_ROOT}/data/**/*.jsonl', recursive=True)):
    print(f'  {os.path.getsize(path) / 1e6:8.1f} MB  {path}')

## 3. Progress so far

Loads no model. Shows every teacher of the config, so you also see what the other notebook has done. A 'lock refreshed N min ago' note means a session is encoding that teacher right now.

In [ ]:
!python -m turkish_distill.encode_teachers --config {CONFIG} {override_args} --status

## 4. GPU check

Qwen3-Embedding-8B in bf16 needs about 16 GB for the weights alone, so it needs a 24 GB GPU (L4 or better); BGE-M3 and mE5 fit on a smaller one.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU: Runtime > Change runtime type.')
gpu = torch.cuda.get_device_properties(0)
print(f'{gpu.name}: {gpu.total_memory / 1e9:.1f} GB')
if 'qwen3_8b' in TEACHERS and gpu.total_memory < 22e9:
    raise RuntimeError('qwen3_8b needs a 24 GB GPU (L4 or better); pick another runtime or another teacher.')

## 5. Encode

Resumable and safe to interrupt: finished shards are kept and skipped. Qwen3-8B takes about 26 minutes per 10,000 texts on an L4 (measured on the Cosmos passages; the stage-1 texts are mostly short questions, so it should be faster).

In [ ]:
lock_arg = '--ignore-lock' if IGNORE_LOCK else ''
!python -m turkish_distill.encode_teachers --config {CONFIG} {override_args} {only_args} {lock_arg}

## 6. Progress after encoding

In [ ]:
!python -m turkish_distill.encode_teachers --config {CONFIG} {override_args} --status